# Serving, policy, and audit

Run notebook 04 first. It prepares the isolated Registry state used here. This notebook demonstrates the actual path: **champion alias -> Model API -> probability -> Policy -> model_evaluation**. Registry state and runtime state are deliberately separate: the API loads its model once at startup.

In [1]:
import json
import os
import socket
import subprocess
import sys
from pathlib import Path
from uuid import uuid4

import httpx
import mlflow
import pandas as pd
from IPython.display import display
from mlflow.tracking import MlflowClient

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DEMO_ROOT = PROJECT_ROOT / 'var' / 't23_5_demo'
STATE_PATH = DEMO_ROOT / 'state.json'
if not STATE_PATH.exists():
    raise RuntimeError('Run notebook 04 first to prepare the isolated demo Registry.')
state = json.loads(STATE_PATH.read_text())
state.setdefault('completed_actions', {})
state.setdefault('evaluations', {})
TRACKING_URI = f"sqlite:///{DEMO_ROOT / 'mlflow.db'}"
DEMO_DB = DEMO_ROOT / 'invoiceops.db'
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()
print(f'Using isolated Registry at {TRACKING_URI}')

Using isolated Registry at sqlite:////Users/admin/Desktop/elements/usach/diplomado/invoice_ops/invoice-ai/var/t23_5_demo/mlflow.db


In [2]:
from invoiceops.ml.registry import MODEL_NAME, promote_model

champion = client.get_model_version_by_alias(MODEL_NAME, 'champion')
champion_run = client.get_run(champion.run_id)
display(pd.DataFrame([{
    'model_name': MODEL_NAME, 'model_version': champion.version, 'run_id': champion.run_id,
    'model_type': champion_run.data.params.get('model_type'),
}]))
print('Registry alias resolved dynamically; no version number is hardcoded.')

,model_name,model_version,run_id,model_type
0,invoice-review,2,dddca7b27b374e7fa3a30d695ee7228d,random_forest


Registry alias resolved dynamically; no version number is hardcoded.


## Local API lifecycle

The server binds only to `127.0.0.1` on an unused local port. Health polling is bounded. The cleanup cell terminates only the process this notebook created, even if an error occurs.

In [3]:
from notebooks._demo_helpers import cleanup_created_process, wait_for_health

api_process = None
api_created_by_notebook = False

def unused_local_port():
    with socket.socket() as listener:
        listener.bind(('127.0.0.1', 0))
        return listener.getsockname()[1]

def start_api():
    global api_process, api_created_by_notebook, BASE_URL
    cleanup_created_process(api_process, created_by_helper=api_created_by_notebook)
    port = unused_local_port()
    BASE_URL = f'http://127.0.0.1:{port}'
    environment = os.environ | {
        'MLFLOW_TRACKING_URI': TRACKING_URI,
        'INVOICEOPS_MODEL_URI': f'models:/{MODEL_NAME}@champion',
        'PYTHONPATH': str(PROJECT_ROOT / 'src'),
    }
    venv_python = PROJECT_ROOT / '.venv' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
    python_executable = str(venv_python) if venv_python.is_file() else sys.executable
    api_process = subprocess.Popen(
        [python_executable, '-m', 'uvicorn', 'invoiceops.model_api.app:app', '--host', '127.0.0.1', '--port', str(port)],
        cwd=PROJECT_ROOT, env=environment, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True,
    )
    api_created_by_notebook = True
    def health_request(url):
        try:
            return httpx.get(url, timeout=1).status_code == 200
        except httpx.RequestError:
            return False
    try:
        wait_for_health(
            f'{BASE_URL}/health', timeout_seconds=20, poll_interval_seconds=0.25,
            request=health_request,
        )
    except Exception:
        cleanup_created_process(api_process, created_by_helper=api_created_by_notebook)
        raise

start_api()
health = httpx.get(f'{BASE_URL}/health', timeout=5).json()
display(pd.DataFrame([health]))

,model_name,model_version,run_id,status
0,invoice-review,2,dddca7b27b374e7fa3a30d695ee7228d,ok


In [4]:
from invoiceops.domain.policy import fallback_recommendation, recommend_from_probability
from invoiceops.domain.rules import RULE_VERSION, decide_invoice
from invoiceops.legacy.db import get_invoice, init_db, insert_model_evaluation, list_model_evaluations
from invoiceops.legacy.seed import seed_invoices
from invoiceops.ml.features import invoice_to_features

init_db(DEMO_DB)
if get_invoice(DEMO_DB, 'INV-10029') is None:
    seed_invoices(DEMO_DB)

def predict_and_recommend(invoice_id):
    invoice = get_invoice(DEMO_DB, invoice_id)
    response = httpx.post(f'{BASE_URL}/predict', json=invoice_to_features(invoice), timeout=5)
    response.raise_for_status()
    prediction = response.json()
    recommendation = recommend_from_probability(prediction['manual_review_probability'])
    return invoice, prediction, recommendation

for invoice_id in ('INV-10029', 'INV-10030'):
    invoice, prediction, recommendation = predict_and_recommend(invoice_id)
    display(pd.DataFrame([{
        'invoice_id': invoice_id, 'rule_version': RULE_VERSION, 'rule_result': decide_invoice(invoice).value,
        'model_probability': prediction['manual_review_probability'], 'policy_version': recommendation.policy_version,
        'policy_recommendation': recommendation.decision.value,
    }]))
    if invoice_id == 'INV-10029':
        assert decide_invoice(invoice).value == 'AUTO_PROCESS'
    else:
        assert decide_invoice(invoice).value == 'AUTO_PROCESS'
        print('INV-10030 uses the real served score; do not fabricate a Policy outcome.')

0 migrations pending


,invoice_id,rule_version,rule_result,model_probability,policy_version,policy_recommendation
0,INV-10029,invoice-rules-v1,AUTO_PROCESS,0.24,ml-policy-v1,AUTO_PROCESS


,invoice_id,rule_version,rule_result,model_probability,policy_version,policy_recommendation
0,INV-10030,invoice-rules-v1,AUTO_PROCESS,0.77,ml-policy-v1,AUTO_PROCESS


INV-10030 uses the real served score; do not fabricate a Policy outcome.


## Compare A and B on the same invoice

Each promotion is followed by an API restart and `/health` verification. The same Policy is applied to both real scores. **WARNING: MODIFIES STATE** cells below persist at most one audit record per isolated demo action.

In [5]:
# WARNING: MODIFIES STATE - promotion and audit actions are independently guarded.
from notebooks._demo_helpers import run_mutable_action_once

def save_state():
    STATE_PATH.write_text(json.dumps(state, indent=2, sort_keys=True) + '\n')

def promote_for_audit(candidate):
    version = str(state['registered_versions'][candidate])
    action = f'promote-for-audit-{candidate}'
    def promote_once():
        current = client.get_model_version_by_alias(MODEL_NAME, 'champion').version
        if current != version:
            promote_model(version)
        return version
    promoted = run_mutable_action_once(action, state['completed_actions'], promote_once)
    save_state()
    return str(promoted)

def serve_and_persist(candidate):
    version = promote_for_audit(candidate)
    action = f'persist-INV-10030-{candidate}'
    def persist_once():
        start_api()  # The runtime changes only after this restart.
        health = httpx.get(f'{BASE_URL}/health', timeout=5).json()
        assert health['model_version'] == version
        invoice, prediction, recommendation = predict_and_recommend('INV-10030')
        correlation_id = f't23-5-{candidate.lower()}-{uuid4()}'
        insert_model_evaluation(
            DEMO_DB, invoice.invoice_id, correlation_id=correlation_id, recommendation=recommendation,
            model_name=prediction['model_name'], model_version=prediction['model_version'],
            run_id=prediction['run_id'], manual_review_probability=prediction['manual_review_probability'],
        )
        return {'model_version': prediction['model_version'], 'model_type': client.get_run(prediction['run_id']).data.params.get('model_type'), 'probability': prediction['manual_review_probability'], 'recommendation': recommendation.decision.value}
    state['evaluations'][candidate] = run_mutable_action_once(action, state['completed_actions'], persist_once)
    save_state()

for candidate in ('A', 'B'):
    serve_and_persist(candidate)
display(pd.DataFrame([state['evaluations']['A'], state['evaluations']['B']], index=['A', 'B']))
display(pd.DataFrame([dict(row) for row in list_model_evaluations(DEMO_DB, 'INV-10030')]))

,model_type,model_version,probability,recommendation
A,random_forest,1,0.77,AUTO_PROCESS
B,random_forest,2,0.77,AUTO_PROCESS


,id,invoice_id,correlation_id,model_name,model_version,run_id,manual_review_probability,policy_version,policy_threshold,recommendation,source,reason,created_at
0,3,INV-10030,t23-5-fallback-df78bee7-d018-4da4-bdd4-5b811d2...,None,None,None,NaN,ml-policy-v1,0.8,MANUAL_REVIEW,fallback,model_unavailable,2026-08-26T08:12:24.422636+00:00
1,2,INV-10030,t23-5-b-56d33a2f-3c69-49e5-acb7-729e30459468,invoice-review,2,dddca7b27b374e7fa3a30d695ee7228d,0.77,ml-policy-v1,0.8,AUTO_PROCESS,model,probability_below_threshold,2026-08-26T08:12:24.131324+00:00
2,1,INV-10030,t23-5-a-e2b2272a-608c-4602-9962-845d28c2e373,invoice-review,1,e2b7cfd8abbb4631afc7f34aebac39b7,0.77,ml-policy-v1,0.8,AUTO_PROCESS,model,probability_below_threshold,2026-08-26T08:12:17.587671+00:00


In [6]:
# WARNING: MODIFIES STATE - persist one explicit fallback audit record with no fake score.
cleanup_created_process(api_process, created_by_helper=api_created_by_notebook)
api_process = None
api_created_by_notebook = False
fallback = fallback_recommendation()
def persist_fallback():
    insert_model_evaluation(
        DEMO_DB, 'INV-10030', correlation_id=f't23-5-fallback-{uuid4()}', recommendation=fallback,
        model_name=None, model_version=None, run_id=None, manual_review_probability=None,
    )
    return {'recommendation': fallback.decision.value, 'source': fallback.source, 'reason': fallback.reason, 'probability': None}
fallback_result = run_mutable_action_once('persist-fallback-INV-10030', state['completed_actions'], persist_fallback)
state['evaluations']['fallback'] = fallback_result
save_state()
assert fallback_result == {'recommendation': 'MANUAL_REVIEW', 'source': 'fallback', 'reason': 'model_unavailable', 'probability': None}
display(pd.DataFrame([fallback_result]))
display(pd.DataFrame([dict(row) for row in list_model_evaluations(DEMO_DB, 'INV-10030')]))

,probability,reason,recommendation,source
0,None,model_unavailable,MANUAL_REVIEW,fallback


,id,invoice_id,correlation_id,model_name,model_version,run_id,manual_review_probability,policy_version,policy_threshold,recommendation,source,reason,created_at
0,3,INV-10030,t23-5-fallback-df78bee7-d018-4da4-bdd4-5b811d2...,None,None,None,NaN,ml-policy-v1,0.8,MANUAL_REVIEW,fallback,model_unavailable,2026-08-26T08:12:24.422636+00:00
1,2,INV-10030,t23-5-b-56d33a2f-3c69-49e5-acb7-729e30459468,invoice-review,2,dddca7b27b374e7fa3a30d695ee7228d,0.77,ml-policy-v1,0.8,AUTO_PROCESS,model,probability_below_threshold,2026-08-26T08:12:24.131324+00:00
2,1,INV-10030,t23-5-a-e2b2272a-608c-4602-9962-845d28c2e373,invoice-review,1,e2b7cfd8abbb4631afc7f34aebac39b7,0.77,ml-policy-v1,0.8,AUTO_PROCESS,model,probability_below_threshold,2026-08-26T08:12:17.587671+00:00


### Optional / advanced: stale runtime

When the API is running, moving `champion` in the Registry does not change `/health` until the process is restarted. The notebook intentionally does not implement hot reload. Registry state is not runtime state.

The final cleanup has already stopped the process started by this notebook. The isolated files live under `var/t23_5_demo/`; delete only that directory when the classroom session is over.